REQUIRED LIBRARIES

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib

LOAD DATASET

In [6]:
excel_file='SupplyChainEmissionFactorsforUSIndustriesCommodities.xlsx'
years=range(2010,2017)

In [7]:
years[0]

2010

In [ ]:
df_1 = pd.read_excel(excel_file, sheet_name=f'{years[0]}_Details_commodity')
df_1.head()

ImportError: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.

In [24]:
all_data = []

for year in years:
    try:
        df_com = pd.read_excel(excel_file, sheet_name=f'{year}_Detail_Commodity')
        df_ind = pd.read_excel(excel_file, sheet_name=f'{year}_Detail_Industry')
        
        df_com['Source'] = 'Commodity'
        df_ind['Source'] = 'Industry'
        df_com['Year'] = df_ind['Year'] = year
        
        df_com.columns = df_com.columns.str.strip()
        df_ind.columns = df_ind.columns.str.strip()

        df_com.rename(columns={
            'Commodity Code': 'Code',
            'Commodity Name': 'Name'
        }, inplace=True)
        
        df_ind.rename(columns={
            'Industry Code': 'Code',
            'Industry Name': 'Name'
        }, inplace=True)
        
        all_data.append(pd.concat([df_com, df_ind], ignore_index=True))
        
    except Exception as e:
        print(f"Error processing year {year}: {e}")

Error processing year 2010: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.
Error processing year 2011: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.
Error processing year 2012: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.
Error processing year 2013: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.
Error processing year 2014: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.
Error processing year 2015: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.
Error processing year 2016: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.


In [ ]:
all_data[3]

IndexError: list index out of range

In [ ]:
len(all_data)

In [ ]:
df = pd.concat(all_data, ignore_index=True)
df.head()

Data Preprocessing

In [ ]:
df.columns # Checking columns

In [ ]:
df.isnull().sum()

In [ ]:

# As there is no data avaialble in Unnamed coulmn so we will drop the column
df.drop(columns=['Unnamed: 7'],inplace=True)

In [ ]:
df.columns

In [ ]:
print(df.info())

In [ ]:
df.describe().T

In [ ]:
df.isnull().sum()

In [ ]:
sns.histplot(df['Supply Chain Emission Factors with Margins'], bins=50, kde=True)
plt.title('Target Variable Distribution')
plt.show()

In [ ]:
print(df['Substance'].value_counts())

In [ ]:
print(df['Unit'].value_counts())

In [ ]:
print(df['Unit'].unique())

In [ ]:
print(df['Source'].value_counts())

In [ ]:
df['Substance'].unique()

In [ ]:
substance_map={'carbon dioxide':0, 'methane':1, 'nitrous oxide':2, 'other GHGs':3}

In [ ]:

df['Substance']=df['Substance'].map(substance_map) 

In [ ]:
df['Substance'].unique()

In [ ]:
print(df['Unit'].unique())

In [ ]:
unit_map={'kg/2018 USD, purchaser price':0, 'kg CO2e/2018 USD, purchaser price':1}

In [ ]:
df['Unit']=df['Unit'].map(unit_map)

In [ ]:
print(df['Unit'].unique())

In [ ]:
print(df['Source'].unique())

In [ ]:
source_map={'Commodity':0, 'Industry':1}

In [ ]:
df['Source']=df['Source'].map(source_map)

In [ ]:
print(df['Source'].unique())

In [ ]:
df.info()

In [ ]:
df.Code.unique()

In [ ]:
df.Name.unique()

In [ ]:
len(df.Name.unique())

In [ ]:
top_emitters = df[['Name', 'Supply Chain Emission Factors with Margins']].groupby('Name').mean().sort_values(
    'Supply Chain Emission Factors with Margins', ascending=False).head(10) 

# Resetting index for better plotting
top_emitters = top_emitters.reset_index()

In [ ]:
top_emitters

In [ ]:
plt.figure(figsize=(10,6))
# Example: Top emitting industries (already grouped)
sns.barplot(
    x='Supply Chain Emission Factors with Margins',
    y='Name',
    data=top_emitters,
    hue='Name',
    palette='viridis'  # Use 'Blues', 'viridis', etc., for other color maps
)

# Add ranking labels (1, 2, 3...) next to bars
for i, (value, name) in enumerate(zip(top_emitters['Supply Chain Emission Factors with Margins'], top_emitters.index), start=1):
    plt.text(value + 0.01, i - 1, f'#{i}', va='center', fontsize=11, fontweight='bold', color='black')

plt.title('Top 10 Emitting Industries', fontsize=14, fontweight='bold') # Title of the plot 
plt.xlabel('Emission Factor (kg CO2e/unit)') # X-axis label
plt.ylabel('Industry') # Y-axis label
plt.grid(axis='x', linestyle='--', alpha=0.6) # Adding grid lines for better readability
plt.tight_layout() # Adjust layout to prevent overlap

plt.show()

In [ ]:
df.drop(columns=['Name','Code','Year'], inplace=True) 

In [ ]:
df.head(1)

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
X = df.drop(columns=['Supply Chain Emission Factors with Margins']) # Feature set excluding the target variable
y = df['Supply Chain Emission Factors with Margins']

In [ ]:
X.head()

In [ ]:

y.head()

In [ ]:

# Count plot for Substance
plt.figure(figsize=(6, 3))
sns.countplot(x=df["Substance"])
plt.title("Count Plot: Substance")
plt.xticks()
plt.tight_layout()
plt.show()


In [ ]:
# Count plot for Unit
plt.figure(figsize=(6, 3))
sns.countplot(x=df["Unit"])
plt.title("Count Plot: Unit")
plt.tight_layout()
plt.show()

In [ ]:
# Count plot for Source
plt.figure(figsize=(6, 4))
sns.countplot(x=df["Source"])
plt.title("Count Plot: Source (Industry vs Commodity)")
plt.tight_layout()
plt.show()

In [ ]:
df.columns

In [ ]:
df.select_dtypes(include=np.number).corr()

In [ ]:
df.info()

In [ ]:
# Correlation matrix 
plt.figure(figsize=(12, 8))
sns.heatmap(df.select_dtypes(include=np.number).corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()